In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "datasets", "wordcloud", "mlflow"])
    from google.colab import drive
    drive.mount("/content/drive")

import os
if IN_COLAB:
    os.chdir("/content/drive/MyDrive/amazon-reviews-sentiment-analysis")
else:
    # Ensure src/ is importable
    sys.path.insert(0, os.path.abspath("..")) if os.path.basename(os.getcwd()) == "notebooks" else None

# Multi-Model Benchmarking with PySpark

In [ ]:
from src.config import ProjectConfig
from src.data_loader import get_spark_session, load_data
from src.models import build_full_pipeline
from src.evaluation import compute_metrics, get_confusion_matrix, plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve, create_comparison_table, plot_comparison_chart
from src.experiment_tracker import init_mlflow, log_experiment
from src.utils import timer

config = ProjectConfig()
spark = get_spark_session(config.spark)
train_df, test_df = load_data(spark, config.data)
train_df.cache()
test_df.cache()
print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

init_mlflow(config.mlflow_tracking_uri, config.mlflow_experiment_name)

## Experiment Grid

In [ ]:
experiments = [
    ("logistic_regression", "count_vectorizer"),
    ("logistic_regression", "tfidf"),
    ("logistic_regression", "ngram_cv"),
    ("logistic_regression", "ngram_tfidf"),
    ("naive_bayes", "count_vectorizer"),
    ("naive_bayes", "tfidf"),
    ("random_forest", "count_vectorizer"),
    ("gbt", "count_vectorizer"),
]

results = []

for model_name, feature_type in experiments:
    run_name = f"{model_name}_{feature_type}"
    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")

    pipeline = build_full_pipeline(feature_type, model_name, config.features)

    with timer(run_name) as elapsed:
        model = pipeline.fit(train_df)
    training_time = elapsed()

    predictions = model.transform(test_df)

    # GBT doesn't support rawPrediction for BinaryClassificationEvaluator
    include_auc = model_name != "gbt"
    metrics = compute_metrics(predictions, include_auc=include_auc)

    result = {
        "model": model_name,
        "feature_type": feature_type,
        **metrics,
        "training_time": training_time,
    }
    results.append(result)

    log_experiment(
        run_name=run_name,
        model_name=model_name,
        feature_type=feature_type,
        hyperparams={"vocab_size": config.features.vocab_size, "min_df": config.features.min_df},
        metrics=metrics,
        training_time=training_time,
    )

    print(f"Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f} | AUC: {metrics.get('auc', 'N/A')}")

## Results Comparison

In [ ]:
comparison = create_comparison_table(results)
print(comparison.to_string(index=False))

In [ ]:
plot_comparison_chart(results, save_path="docs/model_comparison.png")
from IPython.display import Image
Image("docs/model_comparison.png")

## Best Model Analysis

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
best_idx = results_df["accuracy"].astype(float).idxmax()
best = results_df.iloc[best_idx]
print(f"Best model: {best['model']} + {best['feature_type']} (Accuracy: {best['accuracy']})")

# Retrain best model for plots
best_pipeline = build_full_pipeline(best["feature_type"], best["model"], config.features)
best_model = best_pipeline.fit(train_df)
best_predictions = best_model.transform(test_df)

cm = get_confusion_matrix(best_predictions)
plot_confusion_matrix(cm, title=f"Confusion Matrix \u2014 {best['model']}", save_path="docs/confusion_matrix.png")

plot_roc_curve(best_predictions, title=f"ROC Curve \u2014 {best['model']}", save_path="docs/roc_curve.png")
plot_precision_recall_curve(best_predictions, title=f"PR Curve \u2014 {best['model']}", save_path="docs/pr_curve.png")

# Save best model
best_model.write().overwrite().save("app/model_artifacts/spark_model")
print("Best model saved to app/model_artifacts/spark_model")

In [ ]:
spark.stop()
print("Spark session stopped.")